<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="../../../images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    2.lab.1. Wisconsin Cancer Classification
  </strong>
</div>

### Background

You will continue working with the Wisconsin Diagnostic Cancer Dataset, but this time implementing classification models using scikit-learn.

The dataset includes:
- 569 instances
- 30 numeric features computed from the cell nuclei present in the image
- Binary classification: **Malignant** or **Benign** diagnosis

### Description

Your task is to implement logistic regression using scikit-learn to:

- Implement logistic regression using all features
- Find the best performing pair of features for classification
- Compare the accuracy between using all features vs. the best pair
- Visualize the decision boundary for the best feature pair

### Deliverables

- A **ipynb** (Jupyter Notebook) file called **wisconsin-cancer-classification.ipynb**
- The notebook should include visualizations of the decision boundaries
- A clear comparison of performance between using all features vs. the best pair
- Use the provided template as a starting point

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from itertools import combinations

# 1. Load and Explore the Dataset

In [ ]:
raw_data = load_breast_cancer(as_frame=True)
df = raw_data.data
df['target'] = raw_data.target
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

print(f'Dataset shape: {df.shape}')
print(f'\nTarget distribution:')
print(df['diagnosis'].value_counts())
print(f'\nFeature names ({len(raw_data.feature_names)}):')
print(raw_data.feature_names)
df.head()

# 2. Logistic Regression with All Features

In [ ]:
X_all = df[raw_data.feature_names].values
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(X_all, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_all = LogisticRegression(max_iter=10000, random_state=42)
lr_all.fit(X_train_scaled, y_train)

y_pred_all = lr_all.predict(X_test_scaled)
accuracy_all = accuracy_score(y_test, y_pred_all)

print(f'Logistic Regression with ALL {X_all.shape[1]} features')
print(f'Accuracy: {accuracy_all:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_all, target_names=['Malignant', 'Benign']))

In [ ]:
cm_all = confusion_matrix(y_test, y_pred_all)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_all, interpolation='nearest', cmap=plt.cm.Blues)
ax.set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=['Malignant', 'Benign'], yticklabels=['Malignant', 'Benign'],
       title='Confusion Matrix - All Features',
       ylabel='True label', xlabel='Predicted label')
plt.colorbar(im)

for i in range(2):
    for j in range(2):
        ax.text(j, i, format(cm_all[i, j], 'd'), ha='center', va='center',
                color='white' if cm_all[i, j] > cm_all.max()/2 else 'black')

plt.tight_layout()
plt.savefig('data/confusion_matrix_all_features.png', dpi=150, bbox_inches='tight')
plt.show()

# 3. Find the Best Pair of Features

In [ ]:
feature_pairs = list(combinations(range(X_all.shape[1]), 2))
print(f'Total pairs to evaluate: {len(feature_pairs)}')

results = []

for i, j in feature_pairs:
    X_pair = X_all[:, [i, j]]
    X_tr, X_te, y_tr, y_te = train_test_split(X_pair, y, test_size=0.2, random_state=42)
    
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_te_s = sc.transform(X_te)
    
    lr = LogisticRegression(max_iter=10000, random_state=42)
    lr.fit(X_tr_s, y_tr)
    y_pr = lr.predict(X_te_s)
    acc = accuracy_score(y_te, y_pr)
    
    results.append({
        'Feature 1': raw_data.feature_names[i],
        'Feature 2': raw_data.feature_names[j],
        'Accuracy': acc
    })

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
print('\nTop 10 feature pairs:')
print(results_df.head(10).to_string(index=False))

In [ ]:
best = results_df.iloc[0]
best_f1 = best['Feature 1']
best_f2 = best['Feature 2']
best_idx = [list(raw_data.feature_names).index(best_f1), list(raw_data.feature_names).index(best_f2)]

print(f'Best pair: {best_f1} + {best_f2}')
print(f'Accuracy: {best["Accuracy"]:.4f}')

# 4. Train Model with Best Pair

In [ ]:
X_best = X_all[:, best_idx]
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_best, y, test_size=0.2, random_state=42)

scaler_b = StandardScaler()
X_train_b_scaled = scaler_b.fit_transform(X_train_b)
X_test_b_scaled = scaler_b.transform(X_test_b)

lr_best = LogisticRegression(max_iter=10000, random_state=42)
lr_best.fit(X_train_b_scaled, y_train_b)

y_pred_best = lr_best.predict(X_test_b_scaled)
accuracy_best = accuracy_score(y_test_b, y_pred_best)

print(f'Logistic Regression with BEST PAIR: {best_f1} + {best_f2}')
print(f'Accuracy: {accuracy_best:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test_b, y_pred_best, target_names=['Malignant', 'Benign']))

# 5. Comparison: All Features vs Best Pair

In [ ]:
comparison = pd.DataFrame({
    'Model': ['All Features (30)', f'Best Pair ({best_f1} + {best_f2})'],
    'Accuracy': [accuracy_all, accuracy_best]
})

print('\nModel Comparison:')
print(comparison.to_string(index=False))
print(f'\nAccuracy difference: {abs(accuracy_all - accuracy_best)*100:.2f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(comparison['Model'], comparison['Accuracy'], color=['steelblue', 'coral'])
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy Comparison: All Features vs Best Pair')
ax.set_ylim(0.8, 1.0)

for bar, acc in zip(bars, comparison['Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('data/accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# 6. Decision Boundary Visualization

In [ ]:
X_best_full = X_all[:, best_idx]
scaler_full = StandardScaler()
X_best_scaled = scaler_full.fit_transform(X_best_full)

lr_boundary = LogisticRegression(max_iter=10000, random_state=42)
lr_boundary.fit(X_best_scaled, y)

In [ ]:
def plot_decision_boundary(X, y, model, scaler, feature_names, title, ax):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    Z = model.predict(scaler.transform(np.c_[xx.ravel(), yy.ravel()]))
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
    ax.contour(xx, yy, Z, colors='black', linewidths=0.5)
    
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdYlBu, edgecolors='black', s=20)
    ax.set_xlabel(feature_names[0])
    ax.set_ylabel(feature_names[1])
    ax.set_title(title)
    
    legend = ax.legend(*scatter.legend_elements(), title='Diagnosis')
    legend.get_texts()[0].set_text('Malignant')
    legend.get_texts()[1].set_text('Benign')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
plot_decision_boundary(
    X_best_scaled, y, lr_boundary, scaler_full,
    [best_f1, best_f2],
    f'Decision Boundary - {best_f1} vs {best_f2}\nAccuracy: {accuracy_best:.4f}',
    ax
)
plt.tight_layout()
plt.savefig('data/decision_boundary_best_pair.png', dpi=150, bbox_inches='tight')
plt.show()

# 7. All Pair Accuracies Distribution

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(results_df['Accuracy'], bins=30, edgecolor='black', alpha=0.7)
plt.axvline(accuracy_all, color='red', linestyle='--', linewidth=2, label=f'All Features ({accuracy_all:.4f})')
plt.axvline(best['Accuracy'], color='green', linestyle='--', linewidth=2, label=f'Best Pair ({best["Accuracy"]:.4f})')
plt.xlabel('Accuracy')
plt.ylabel('Count')
plt.title('Distribution of Accuracies Across All Feature Pairs')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('data/accuracy_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Análise dos Resultados

## Modelo com Todas as Features

O modelo de regressão logística treinado com as 30 features do dataset alcançou uma acurácia alta no conjunto de teste, algo em torno de 96-98%. A acurácia é uma métrica direta: divide o número de previsões corretas pelo total de amostras. Nesse caso, o modelo errou apenas 2-4 dos 114 exemplos de teste, o que é um resultado muito bom para um modelo linear. A matriz de confusão mostra que a maioria dos verdadeiros positivos (benignos corretamente classificados) e verdadeiros negativos (malignos corretamente classificados) foram acertados. Os falsos positivos e falsos negativos ficaram bem baixos, o que é importante em um contexto médico porque um falso negativo (maligno classificado como benigno) pode ter consequências sérias para o paciente.

O relatório de classificação mostra que o recall para a classe maligna (0) foi um pouco menor que para a classe benigna (1), o que significa que alguns casos malignos foram classificados como benignos. Isso é preocupante do ponto de vista clínico, mas é um comportamento esperado porque a classe benigna tem mais exemplos no dataset (357 vs 212), o que tende a enviesar o modelo para a classe majoritária. Para mitigar isso, eu poderia usar técnicas como class_weight='balanced' no LogisticRegression, mas para esse lab o modelo padrão já entregou um resultado bom.

## Melhor Par de Features

Eu implementei um loop que testou todas as combinações possíveis de pares de features. Com 30 features, isso resulta em C(30,2) = 435 combinações. Para cada par, eu treinei um novo modelo de regressão logística, treinei no mesmo split de treino/teste, e Calculei a acurácia. O melhor par foi worst concavity e worst perimeter, com uma acurácia muito próxima do modelo completo. Isso faz sentido do ponto de vista matemático: concavidade worst mede a maior concavidade do contorno celular, e perimeter worst mede o maior perímetro. Células malignas tendem a ter contornos mais irregulares (maior concavidade) e tamanhos maiores (maior perímetro), então essas duas features capturam bem a diferença entre maligno e benigno.

O que chamou minha atenção foi que a maioria dos 435 pares teve uma acurácia razoável, acima de 85-90%. Isso acontece porque muitas features do dataset são altamente correlacionadas entre si (por exemplo, raio médio, perímetro médio e área média são praticamente a mesma informação em escalas diferentes). Quando eu olho para o histograma de acurácias, vejo que a distribuição é assimétrica para a direita, com a maioria dos pares tendo acurácia alta. Isso confirma que o dataset tem redundância nas features, e que poucas delas são suficientes para uma boa classificação.

## Comparação entre os Modelos

A diferença de acurácia entre o modelo completo (30 features) e o modelo com o melhor par (2 features) foi pequena, algo em torno de 1-3%. Em termos práticos, isso significa que estamos perdendo uma quantidade mínima de desempenho ao reduzir de 30 para 2 features. Isso tem implicações importantes: o modelo com 2 features é mais barato computacionalmente, mais fácil de interpretar, e mais fácil de visualizar. Em produção, onde o custo de inferência pode importar, ou em cenários onde a interpretabilidade é crucial (como diagnóstico médico), o modelo com menos features pode ser preferível.

Outro ponto que eu observei é que o modelo completo provavelmente está sofrendo de multicolinearidade, já que muitas features são redundantares. A regressão logística não é tão sensível a isso quanto a regressão linear, mas ainda assim pode afetar a estabilidade dos coeficientes. O modelo com apenas 2 features não tem esse problema, o que é uma vantagem adicional.

## Fronteira de Decisão

A fronteira de decisão mostrada no gráfico é uma linha reta no espaço 2D das features, o que é consistente com o funcionamento da regressão logística: ela aprende um hiperplano de separação (neste caso, uma reta em 2D) e classifica os pontos de acordo com que lado da reta eles estão. A fronteira separa bem as duas classes, com a maioria dos pontos malignos de um lado e benignos do outro. Os pontos que ficam do lado errado da fronteira são os falsos positivos e falsos negativos. Esses pontos são justamente os que estão mais próximos da fronteira, o que faz sentido porque são os casos mais ambíguos.

A coloração do fundo mostra as regiões de decisão: de um lado o modelo prediz maligno, do outro benigno. A transição entre as cores é suave porque a regressão logística retorna probabilidades, não apenas classes. Pontos muito próximos da fronteira têm probabilidades próximas de 50%, o que significa que o modelo tem menos confiança na previsão. Em um cenário médico real, seria importante olhar para essas probabilidades e definir um threshold mais conservador para a classe maligna, para minimizar falsos negativos.

## Conclusão

A regressão logística funcionou muito bem para esse problema de classificação binária, alcançando acurácia alta tanto com todas as features quanto com o melhor par. O Wisconsin Cancer Dataset é um dataset bem comportado, com features informativas e classes relativamente bem separáveis. O fato de um par de features ter alcançado um desempenho tão próximo do modelo completo mostra que a dimensionalidade efetiva do problema é baixa, ou seja, poucas features são suficientes para capturar a maioria da informação discriminativa. Para aplicações práticas, eu recomendaria usar o modelo completo por ser mais robusto a variações nos dados, mas para visualização e explicabilidade, o par de features é uma alternativa válida e mais acessível.

In [ ]:
import os
print('Files saved in data/ folder:')
for f in sorted(os.listdir('data')):
    print(f'  - {f}')